# Task 4 - Soluzione del sistema lineare

Questo notebook legge i file `A.txt` e `rhs.txt` prodotti dalla Task 3,
costruisce la matrice sparsa, calcola la fattorizzazione di Cholesky
e risolve il sistema lineare.

In [21]:
import numpy as np

## Lettura della matrice

Il file `A.txt` contiene, per ogni elemento non nullo, l'indice di riga,
l'indice di colonna e il valore della matrice.

In [22]:
dati_A = np.loadtxt("../task3/A.txt")

print(dati_A)

[[ 2.    2.   -0.36]
 [ 1.    1.   -0.36]
 [ 3.    3.   -0.36]
 [ 0.    0.   -0.36]
 [ 2.    1.    0.09]
 [ 1.    2.    0.09]
 [ 2.    3.    0.09]
 [ 3.    2.    0.09]
 [ 1.    0.    0.09]
 [ 0.    1.    0.09]
 [ 3.    0.    0.09]
 [ 0.    3.    0.09]]


Leggo il vettore rhs.txt dalla task 3 e creo un vettore Numpy

In [23]:
rhs = np.loadtxt("../task3/rhs.txt")

print(rhs)

[-0.00013791 -0.00386591 -0.1083685  -0.00386591]


A questo punto nella memoria del notebook ho dati_A con le tre colonne di A.txt e poi ho rhs vettore con ii valori di rhs.txt

Devo separare le righe dalle colonne e dai valore di A.txt per poter costruire la matrice

In [31]:
righe = dati_A[:, 0].astype(int)
colonne = dati_A[:, 1].astype(int)
valori = dati_A[:, 2]

print("Righe:", righe)
print("Colonne:", colonne)
print("Valori:", valori)

Righe: [2 1 3 0 2 1 2 3 1 0 3 0]
Colonne: [2 1 3 0 1 2 3 2 0 1 0 3]
Valori: [-0.36 -0.36 -0.36 -0.36  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09]


## Costruzione della matrice sparsa

Gli indici di riga, gli indici di colonna e i valori vengono utilizzati
per costruire la matrice sparsa in formato CSC.(Compressed Sparse Column)

In [32]:
from scipy.sparse import csc_matrix 

dimensione = len(rhs)

A = csc_matrix(
    (valori, (righe, colonne)),
    shape=(dimensione, dimensione)
)

print(A)

<Compressed Sparse Column sparse matrix of dtype 'float64'
	with 12 stored elements and shape (4, 4)>
  Coords	Values
  (0, 0)	-0.36000000000000004
  (1, 0)	0.09000000000000001
  (3, 0)	0.09000000000000001
  (0, 1)	0.09000000000000001
  (1, 1)	-0.36000000000000004
  (2, 1)	0.09000000000000001
  (1, 2)	0.09000000000000001
  (2, 2)	-0.36000000000000004
  (3, 2)	0.09000000000000001
  (0, 3)	0.09000000000000001
  (2, 3)	0.09000000000000001
  (3, 3)	-0.36000000000000004


In [33]:
print("Dimensioni della matrice:", A.shape)
print("Numero di elementi non nulli:", A.nnz)
print("Matrice completa:")
print(A.toarray()) #per dimensioni piccole ok

Dimensioni della matrice: (4, 4)
Numero di elementi non nulli: 12
Matrice completa:
[[-0.36  0.09  0.    0.09]
 [ 0.09 -0.36  0.09  0.  ]
 [ 0.    0.09 -0.36  0.09]
 [ 0.09  0.    0.09 -0.36]]


## Fattorizzazione di Cholesky

Poiché la matrice `A` è definita negativa, viene fattorizzata la matrice
`-A`, ottenendo `-A = L L^T`, con `L` triangolare inferiore.

In [34]:
from sksparse.cholmod import cholesky

def calcola_L(matrice):
    factor = cholesky(
        matrice,
        ordering_method="natural"
    )

    return factor.L()

In [ ]:
L = calcola_L(-A)

print("Matrice L in formato sparso:")
print(L)

## Soluzione del sistema lineare

Il sistema viene risolto mediante due sistemi triangolari:

1. `L y = -rhs`;
2. `L^T u = y`.

In [43]:
from scipy.sparse.linalg import spsolve_triangular

y = spsolve_triangular(L, -rhs,lower=True)

u = spsolve_triangular(L.T,y,lower=False)

print("Soluzione u:")
print(u)

Soluzione u:
[0.05777664 0.1147871  0.35841717 0.1147871 ]


In [44]:
#Calcolo del residuo verifica del sistema
residuo = A @ u - rhs

print("Residuo:")
print(residuo)

print("Norma del residuo:")
print(np.linalg.norm(residuo))

Residuo:
[ 1.73472348e-18 -4.33680869e-19 -1.38777878e-17 -4.33680869e-19]
Norma del residuo:
1.3999229199105634e-17
